# Вкладена часова перевірка на train

Дослідження доповнює ноутбуки 03–04; validation і test не відкриваються. Єдиний вхід — `outputs/train_audit.csv`: з початкових полів аудиту train відновлюється до обробки сум. Правила очищення сум навчаються заново в кожному часовому зрізі, включно з внутрішніми.

Мета — дослідити регуляризацію ЛР і порівняти некалібровані й калібровані ЛР та бустинг, а також ансамбль із рівними вагами за однаковою схемою вкладеної часової перевірки. Дослідження слугує для вибору стратегії і не є незалежною перевіркою, оскільки ці тижні train аналізувалися раніше.

Запуск: `uv sync`, потрібна тека `outputs/`, створена ноутбуком 02; Restart and Run All. Дані, прогнози з ID і виводи ноутбука зберігаються лише локально.

In [ ]:
import matplotlib
import pandas as pd

from disputes.paths import PROJECT as PROJECT_DIR

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

from disputes.train_only_study import freeze_protocol, run

# Freeze the protocol before any data is read; an existing protocol must match and is never overwritten.
OUTPUT, protocol_hash = freeze_protocol(PROJECT_DIR)
print("SHA-256 зафіксованого протоколу:", protocol_hash)
display(pd.DataFrame({"outer_train": ["0–5", "0–9", "0–13"],
                      "outer_evaluation": ["6–9", "10–13", "14–17"],
                      "inner_train_to_evaluation": ["0–1 → 2–3; 0–3 → 4–5",
                          "0–3 → 4–6; 0–6 → 7–9", "0–5 → 6–9; 0–9 → 10–13"]}))
print("Практична близькість: 50 USD на 1 000 оцінених спорів")

## Правило вибору, зафіксоване до отримання результатів

Основний критерій — сумарний Score на спорах із сумою, придатною для Score. Практично близькими вважаються стратегії, Score яких відстає від найбільшого не більше ніж на 0,05 USD × кількість оцінених спорів. Цей поріг є робочим припущенням щодо достатньої користі від ускладнення, а не статистичним критерієм.

Серед близьких стратегій обирається простіша в порядку: стала ймовірність train → ЛР → калібрована ЛР → бустинг → калібрований бустинг → ансамбль. За однакової складності порівнюються послідовно: найменший за періодами приріст відносно сталої ймовірності на один спір (більший кращий), стандартне відхилення цього приросту (менше краще), pooled log loss (менший кращий), назва стратегії. Складність визначено зафіксованим порядком, а не числовою мірою.

Результат без відновлених і виправлених сум є перевіркою чутливості і не використовується для зміни правила вибору.

## Вкладений вибір параметрів і калібрування

Для кожного зовнішнього періоду train два внутрішні зрізи порівнюють `C` = 0,01; 0,1; 1; 10; 100 для ЛР і шість фіксованих конфігурацій бустингу. Обирається найбільший сумарний внутрішній Score; за рівності — менший pooled log loss, потім менше `C` або назва конфігурації. Попередня обробка навчається заново; мітки зовнішнього періоду для вибору параметрів не використовуються.

Для кожної обраної моделі збираються її позачасові прогнози на внутрішніх пізніших підперіодах; на них і відповідних мітках навчається sigmoid-калібратор. Модель, що дала кожен такий прогноз, навчена лише на ранніших тижнях; мітки зовнішнього періоду калібратору недоступні. Внутрішні прогнози використовуються і для вибору параметрів, і для калібрування, тому незалежною оцінкою є лише результат на зовнішньому зрізі.

Калібрування застосовується за наявності щонайменше 1000 прогнозів, 50 прикладів кожного класу і двох внутрішніх періодів. Це евристичний критерій обсягу, а не гарантія якості калібрування. За недостатнього обсягу використовуються некалібровані прогнози, випадок позначається, а калібрований варіант виключається з вибору.

Ансамбль — середнє некаліброваних налаштованих ЛР і бустингу з рівними вагами. Після збору прогнозів для калібрування базові моделі перенавчаються на всьому зовнішньому train; відповідність калібратора цим моделям оцінюється на зовнішньому зрізі.

Окрема сітка `C` на всіх трьох зовнішніх періодах є діагностикою чутливості і не замінює вкладеного вибору, у якому `C` визначається лише за минулими даними. Скрізь застосовується правило рішення p × 0,2 × amount_usd > 6.

In [ ]:
summary, periods, C_grid, calibration, selection = run(PROJECT_DIR)
display(C_grid.round(5))
print("C: результати кожного періоду")
display(pd.read_csv(OUTPUT / "C_grid_periods.csv").round(5))
print("Параметри, вибрані всередині кожної зовнішньої навчальної частини")
display(pd.read_csv(OUTPUT / "outer_choices.csv"))
print("Достатність даних калібрування")
display(calibration)

## Зведена таблиця стратегій

Кожен рядок описує повний алгоритм: внутрішній вибір параметрів → калібрування (за потреби) → навчання базової моделі → прогноз наступного періоду. Класифікаційні метрики обчислюються на всіх зовнішніх спорах, включно зі спорами з невідомою сумою; Score — лише на однакових спорах із сумою, придатною для Score (кількість наведено окремо).

Pooled log loss і Brier обчислюються за об'єднаними зовнішніми прогнозами, а не як незважене середнє зрізів. Pooled ROC-AUC додатково враховує ранжування між періодами; ROC-AUC кожного періоду наведено окремо. Колонки `observed_*` виключають відновлені й виправлені суми; підмножини основного й observed-сценаріїв різні, у межах сценарію — однакові для всіх моделей. Невідомі суми не замінюються нулями.

In [ ]:
columns = ["strategy", "score_usd", "delta_always_usd", "delta_constant_usd",
           "score_weeks_6-9", "score_weeks_10-13", "score_weeks_14-17",
           "log_loss", "brier", "roc_auc", "observed_score_usd",
           "observed_delta_always_usd", "observed_delta_constant_usd",
           "evaluated_rows", "observed_rows", "classification_rows", "selectable"]
display(summary[columns].round(5))
display(periods.round(5))
print("Вибір за протоколом:", selection["selected_strategy"])
print("Формальний максимум Score:", selection["maximum_score_strategy"])
print("Поріг практичної близькості, USD:", selection["practical_margin_usd"])
print("Відставання обраного варіанта від максимуму, USD:", selection["selected_gap_to_max_usd"])
print("Близькі варіанти:", selection["near_best_strategies"])
if not calibration.adequate_by_protocol.all():
    print("УВАГА: є періоди з недостатнім обсягом калібрування — див. status.")
else:
    print("Усі калібратори пройшли евристичне правило обсягу; це не гарантує точного калібрування.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for c, group in pd.read_csv(OUTPUT / "C_grid_periods.csv").groupby("C"):
    axes[0].plot(group.outer_period, group.delta_constant_usd / group.evaluated_rows,
                 "o-", label=f"C={c:g}")
axes[0].set(title="Fixed C sensitivity", xlabel="Outer evaluation weeks",
            ylabel="Uplift vs train-rate policy, USD/dispute")
axes[0].legend(fontsize=8)
for name, group in periods.groupby("strategy", sort=False):
    if name != "constant_train_rate":
        axes[1].plot(group.outer_period, group.delta_constant_usd / group.evaluated_rows,
                     "o-", label=name)
axes[1].set(title="Nested strategy comparison", xlabel="Outer evaluation weeks",
            ylabel="Uplift vs train-rate policy, USD/dispute")
axes[1].legend(fontsize=7)
fig.tight_layout()
figure = OUTPUT / "train_only_comparison.png"
fig.savefig(figure, dpi=150)
display(fig)
plt.close(fig)

## Обмеження

- Протокол обирає стратегію, але не встановлює статистично значущої переваги. Трьох зовнішніх періодів недостатньо для висновків про стійкість.
- Ці дані train раніше використовувалися для огляду даних і попереднього порівняння. Вкладена схема усуває локальний підбір параметрів і калібрування, але не попереднє ознайомлення з даними та вибір простору кандидатів.
- Validation раніше використовувалася для вибору; це дослідження не відновлює її незалежності. Файли validation і test тут не відкривалися.
- Дата отримання результату спору відсутня, тому не перевірено, чи були навчальні мітки доступні на момент наступного прогнозу. Це обмеження стосується всіх часових перевірок у роботі.
- Вибірка містить лише спори, за які боролися й отримали результат. Відновлення сум і частка спорів із сумою, придатною для Score, також обмежують узагальнення.

Ноутбук не оцінює validation чи test і не змінює порогу практичної близькості після перегляду результатів.

`protocol.json` містить зафіксоване правило, `selection.json` — вибір і параметри кожного зовнішнього зрізу, `run_evidence.json` — хеші коду й результатів і журнал фактично відкритих CSV. Аудит-хук Python блокує відкриття інших таблиць `outputs/`, а також CSV набору даних, validation, test і `unknown_time`.

Метод калібрування та вимога відокремлення прогнозованих прикладів від навчання базової моделі: [документація scikit-learn](https://scikit-learn.org/stable/modules/calibration.html).